# Lesson 3: Inside Jev & The Three Typed Primitives

Ask Jev one question of each type, then several in one call, then compare Jev with two frontier LLMs on the same ticket.

Set up the repo first by following **Setup** in the root `README.md`. Populate the repository-root `.env`; the first code cell loads it. The Jev cells need `TYPESAFE_API_KEY`. The final benchmark also needs `OPENAI_API_KEY` and `ANTHROPIC_API_KEY`.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

env_file = Path("../.env")  # notebook kernels start in the lesson folder
if not env_file.is_file():
    raise FileNotFoundError("Create .env in the course repository root. See README.md Setup.")
load_dotenv(env_file, override=True)

from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

client = TypeSafeClient()  # reads TYPESAFE_API_KEY

## Primitive 1: `Noul` (Binary Probability)

Asks a yes/no question about the input (the `state`). Returns `noul`: the probability of yes, from `0.0` to `1.0`.

In [ ]:
res = client.system_one(
    state="rm -rf /var/log/* && git push --force origin main",
    questions={"destructive": Noul(
        instructions="Is this bash command destructive, or does it bypass safety controls?",
    )},
)

print(f"Destructive probability: {res.answers['destructive'].noul:.0%}")

## Primitive 2: `Choice` (Categorical Distribution)

Picks one option. `criteria` maps each option to a description. Returns `choice`, `probabilities` for every option, and `confidence`.

In [ ]:
res = client.system_one(
    state="Pods in our Kubernetes cluster are showing CrashLoopBackOff",
    questions={
        "department": Choice(
            instructions="Which department should handle this ticket?",
            criteria={
                "billing": "Charges, refunds, invoices",
                "infrastructure": "Servers, clusters, outages",
                "sales": "Pricing, upgrades, new seats",
                "security": "Attacks, leaked credentials",
            },
        ),
    },
)

dept = res.answers["department"]
print(f"Department: {dept.choice} (confidence {dept.confidence:.2f})")

In [ ]:
dept.probabilities

## Primitive 3: `Score` (Ordinal Rating)

Rates the input on ordered levels you describe. Returns `score` (a position from `0` to the top level), `probabilities`, and `confidence`.

In [ ]:
res = client.system_one(
    state="The entire checkout API is returning 500 errors during Black Friday",
    questions={"urgency": Score(
        instructions="How urgent is this incident?",
        criteria=["Low", "Medium", "High", "Critical"],  # levels 0-3
    )},
)

urgency = res.answers["urgency"]
print(f"Urgency: {urgency.score:.2f} / 3 (confidence {urgency.confidence:.2f})")

In [ ]:
urgency.probabilities

## Three Questions, One Call

One call can mix `Choice`, `Score`, and `Noul` questions. Each question is evaluated independently against the same `state`.

In [ ]:
res = client.system_one(
    state="Our production cluster has been down for an hour and customers can't check out.",
    questions={
        "department": Choice(
            instructions="Which department should handle this ticket?",
            criteria={
                "billing": "Charges, refunds, invoices",
                "infrastructure": "Servers, clusters, outages",
                "sales": "Pricing, upgrades, new seats",
                "security": "Attacks, leaked credentials",
            },
        ),
        "urgency": Score(
            instructions="How urgent is this ticket?",
            criteria=["Low", "Medium", "High", "Critical"],
        ),
        "spam": Noul(
            instructions="Is this ticket unsolicited advertising or spam?",
        ),
    },
)

answers = res.answers
print(f"Department: {answers['department'].choice} (confidence {answers['department'].confidence:.2f})")
print(f"Urgency: {answers['urgency'].score:.2f} / 3")
print(f"Spam probability: {answers['spam'].noul:.0%}")

## Benchmark: Jev vs LLMs on One Ticket

Routes one support ticket with GPT-6 Sol, Claude Opus 5.5, and Jev. All three make the same three decisions: department, urgency, and whether the ticket can be auto-resolved.

Costs use each API's reported token usage at standard uncached USD rates, verified 2026-09-30. Latency is wall-clock time from your machine, so it includes the network round trip and varies from run to run.

In [ ]:
import time
from typing import Literal

import anthropic
import openai
from pydantic import BaseModel

TICKET = (
    "Hello, I was charged twice for my subscription renewal on September 15th. "
    "Can you please issue a refund for the duplicate transaction?"
)

DEPARTMENTS = {
    "billing": "Charges, refunds, invoices, subscriptions",
    "technical": "Bugs, errors, integrations",
    "sales": "Pricing, upgrades, new seats",
    "security": "Attacks, leaked credentials, account compromise",
}
URGENCY_LEVELS = ["Routine", "Standard", "Blocking", "Outage"]

# USD per 1M tokens (input, output), standard uncached rates verified 2026-09-30.
PRICES = {
    "gpt-6-sol": (2.00, 10.00),
    "claude-opus-5-5": (4.00, 20.00),
    "jev": (0.042, 0.00),
}


def cost(model: str, input_tokens: int, output_tokens: int) -> float:
    input_price, output_price = PRICES[model]
    return (input_tokens * input_price + output_tokens * output_price) / 1_000_000

### The LLMs: one structured-output call each

In [ ]:
LLM_PROMPT = f"""Route this support ticket.

Departments: {DEPARTMENTS}
Urgency levels: {URGENCY_LEVELS}
auto_resolve: true if this is a refund request for a duplicate charge.

Ticket: {TICKET}"""


class Routing(BaseModel):
    department: Literal["billing", "technical", "sales", "security"]
    urgency: Literal["Routine", "Standard", "Blocking", "Outage"]
    auto_resolve: bool


def route_with_gpt() -> dict:
    client = openai.OpenAI()
    start = time.perf_counter()
    response = client.responses.parse(
        model="gpt-6-sol",
        input=LLM_PROMPT,
        text_format=Routing,
        reasoning={"effort": "low"},
    )
    latency_ms = (time.perf_counter() - start) * 1000
    usage = response.usage
    return {
        "name": "GPT-6 Sol",
        "answer": response.output_parsed.model_dump(),
        "latency_ms": latency_ms,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "cost_usd": cost("gpt-6-sol", usage.input_tokens, usage.output_tokens),
    }


def route_with_claude() -> dict:
    client = anthropic.Anthropic()
    start = time.perf_counter()
    response = client.messages.parse(
        model="claude-opus-5-5",
        max_tokens=2000,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": LLM_PROMPT}],
        output_format=Routing,
    )
    latency_ms = (time.perf_counter() - start) * 1000
    if response.stop_reason != "end_turn":
        raise RuntimeError(f"Claude stopped with {response.stop_reason}")
    usage = response.usage
    return {
        "name": "Claude Opus 5.5",
        "answer": response.parsed_output.model_dump(),
        "latency_ms": latency_ms,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "cost_usd": cost("claude-opus-5-5", usage.input_tokens, usage.output_tokens),
    }

### Jev: one call, three typed questions

In [ ]:
def route_with_jev() -> dict:
    start = time.perf_counter()
    response = client.system_one(
        state=TICKET,
        questions={
            "department": Choice(
                instructions="Which department should handle this ticket?",
                criteria=DEPARTMENTS,
            ),
            "urgency": Score(
                instructions="How urgent is this ticket?",
                criteria=URGENCY_LEVELS,
            ),
            "auto_resolve": Noul(
                instructions="Is this a refund request for a duplicate charge?",
            ),
        },
    )
    latency_ms = (time.perf_counter() - start) * 1000
    answers = response.answers
    usage = response.usage
    return {
        "name": "Jev",
        "answer": {
            "department": answers["department"].choice,
            "urgency": URGENCY_LEVELS[round(answers["urgency"].score)],
            "auto_resolve": answers["auto_resolve"].noul >= 0.5,
        },
        "latency_ms": latency_ms,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "cost_usd": cost("jev", usage.input_tokens, usage.output_tokens),
    }

### Run all three

In [ ]:
print(f'Ticket: "{TICKET}"\n')

results = [route_with_gpt(), route_with_claude(), route_with_jev()]
for r in results:
    print(f"{r['name']:<16} {r['latency_ms']:>7,.0f} ms   ${r['cost_usd']:.6f}   "
          f"{r['input_tokens']} in / {r['output_tokens']} out   {r['answer']}")

jev = results[-1]
print()
for r in results[:-1]:
    print(f"Jev vs {r['name']}: {r['latency_ms'] / jev['latency_ms']:.0f}x faster, "
          f"{r['cost_usd'] / jev['cost_usd']:.0f}x cheaper")